# Frozen four-treatment task-aware pilot

This notebook runs the bounded task-treatment gate on the verified prompt-neutral GLIM vectors. Before running: enable a Kaggle GPU, enable Internet, enable the private `GITHUB_TOKEN` secret, and attach exact dataset version 2 of `thestonedape/task-aware-eegtotext`. You may also attach one prior partial pilot-output dataset to resume completed configuration/seed units.

The all-four real-data smoke freezes the deterministic 24-way pool SHA before the full 12-run matrix. The held-out test split is never available to this runner.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = '8c6a0065fcc34f73d354c9d8ca0dbddae801b99b'
WORKTREE = '/kaggle/working/SemKey'
PRESERVED_SOURCE_ID = 'kaggle-dataset-thestonedape-task-aware-eegtotext-version-2'
EXPECTED_INPUT_SHA256 = '6c1fff8d2e89e33a72d03c39651e8ecce678c3b93cdb66747dd6dcc00538cddb'
EXPECTED_PROTOCOL_SHA256 = '35519ef55af615e593000bc353ad1e6d7043238352ad7f83b8f23c3aa1ddd9b7'
SMOKE_OUTPUT = '/kaggle/working/task-treatment-pilot-smoke'
OUTPUT = '/kaggle/working/task-aware-eeg2text-task-treatment-pilots'
CONFIGURATIONS = ['generic_pooled', 'separate_per_task', 'task_token', 'masked_shared_private']
SEEDS = [20260717, 20260718, 20260719]
assert len(COMMIT) == 40
assert all(len(value) == 64 for value in (EXPECTED_INPUT_SHA256, EXPECTED_PROTOCOL_SHA256))

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
import numpy as np
import torch
from kaggle_secrets import UserSecretsClient

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
versions = {
    'python': platform.python_version(), 'numpy': np.__version__,
    'torch': torch.__version__, 'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0),
}
print(versions)

def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()

github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == COMMIT
protocol_path = os.path.join(WORKTREE, 'evaluation', 'task_treatment_pilot_execution_protocol.json')
assert digest(protocol_path) == EXPECTED_PROTOCOL_SHA256
subprocess.run([
    sys.executable, '-m', 'unittest',
    'evaluation.test_task_treatment_pilot_runner',
    'project_adapters.test_task_treatment_pilots',
    'evaluation.test_task_treatment_pilot_contract',
    'evaluation.test_verify_prompt_neutral_pilot_inputs',
], check=True, cwd=WORKTREE)
print({'project_commit': actual_commit, 'protocol_sha256': EXPECTED_PROTOCOL_SHA256, 'regressions': 'PASS'})

In [ ]:
manifest_candidates = glob.glob('/kaggle/input/**/pilot_input_manifest.json', recursive=True)
artifact_roots = []
required = [
    'eeg/vector_manifest.json', 'eeg/vector_index.csv',
    'text/text_vector_manifest.json', 'text/text_vector_index.csv',
    'text/trial_text_targets.csv', 'run_metadata.json',
]
for path in manifest_candidates:
    root = os.path.dirname(path)
    if all(os.path.isfile(os.path.join(root, item)) for item in required):
        artifact_roots.append(root)
assert len(artifact_roots) == 1, ('Attach exact source dataset version 2 once', artifact_roots, manifest_candidates)
ARTIFACT_ROOT = artifact_roots[0]
assert digest(os.path.join(ARTIFACT_ROOT, 'pilot_input_manifest.json')) == EXPECTED_INPUT_SHA256

resume_roots = []
for marker in glob.glob('/kaggle/input/**/frozen_candidate_pools.csv', recursive=True):
    root = os.path.dirname(marker)
    if os.path.isdir(os.path.join(root, 'runs')):
        resume_roots.append(root)
resume_roots = sorted(set(resume_roots))
assert len(resume_roots) <= 1, ('Attach at most one prior pilot output', resume_roots)
if not os.path.exists(OUTPUT) and resume_roots:
    print({'resume_from': resume_roots[0]})
    shutil.copytree(resume_roots[0], OUTPUT)
print({'artifact_root': ARTIFACT_ROOT, 'input_manifest_sha256': EXPECTED_INPUT_SHA256, 'resume_output': bool(resume_roots)})

In [ ]:
def pilot_command(output, smoke=False, expected_pool_sha256=None):
    command = [
        sys.executable, os.path.join(WORKTREE, 'evaluation', 'run_task_treatment_pilots.py'),
        '--artifact-root', ARTIFACT_ROOT,
        '--output-root', output,
        '--preserved-source-id', PRESERVED_SOURCE_ID,
        '--project-commit', actual_commit,
        '--device', 'cuda',
    ]
    if smoke:
        command.append('--smoke')
    if expected_pool_sha256 is not None:
        command += ['--expected-candidate-pool-sha256', expected_pool_sha256]
    return command

if os.path.exists(SMOKE_OUTPUT):
    shutil.rmtree(SMOKE_OUTPUT)
subprocess.run(pilot_command(SMOKE_OUTPUT, smoke=True), check=True)
smoke = json.load(open(os.path.join(SMOKE_OUTPUT, 'smoke_report.json'), encoding='utf-8'))
assert smoke['status'] == 'pass' and smoke['run_mode'] == 'smoke'
assert smoke['runs'] == 4 and smoke['configurations'] == CONFIGURATIONS
assert smoke['held_out_test_accessed'] is False
POOL_SHA256 = smoke['candidate_pool_sha256']
assert len(POOL_SHA256) == 64
for config_id in CONFIGURATIONS:
    summary_path = os.path.join(SMOKE_OUTPUT, 'runs', config_id, str(SEEDS[0]), 'run_summary.json')
    summary = json.load(open(summary_path, encoding='utf-8'))
    binding = summary['binding']
    assert summary['status'] == 'pass' and summary['held_out_test_accessed'] is False
    assert binding['project_commit'] == actual_commit
    assert binding['preserved_source_id'] == PRESERVED_SOURCE_ID
    assert binding['candidate_pool_sha256'] == POOL_SHA256
print({'all_four_real_data_smoke': 'PASS', 'candidate_pool_sha256': POOL_SHA256, 'validation_partition_sha256': smoke['validation_partition_sha256']})

In [ ]:
# Long cell. Completed configuration/seed units are hash-verified and reused.
# If Kaggle interrupts the run, save OUTPUT as a private dataset and attach it on the next run.
subprocess.run(pilot_command(OUTPUT, expected_pool_sha256=POOL_SHA256), check=True)

In [ ]:
manifest_path = os.path.join(OUTPUT, 'pilot_manifest.json')
manifest = json.load(open(manifest_path, encoding='utf-8'))
assert manifest['status'] == 'pass' and manifest['run_mode'] == 'full'
assert manifest['configurations'] == CONFIGURATIONS and manifest['seeds'] == SEEDS
assert manifest['project_commit'] == actual_commit
assert manifest['execution_protocol_sha256'] == EXPECTED_PROTOCOL_SHA256
assert manifest['input_manifest_sha256'] == EXPECTED_INPUT_SHA256
assert manifest['preserved_source_id'] == PRESERVED_SOURCE_ID
assert manifest['candidate_pool_sha256'] == POOL_SHA256
assert manifest['held_out_test_accessed'] is False
for relative, expected in manifest['artifact_sha256'].items():
    assert digest(os.path.join(OUTPUT, *relative.split('/'))) == expected, relative
assert len(manifest['run_summary_sha256']) == len(CONFIGURATIONS) * len(SEEDS)
required_run_artifacts = {'best_checkpoint.pt', 'training_history.csv', 'predictions.csv', 'metrics.csv'}
for relative, expected in manifest['run_summary_sha256'].items():
    path = os.path.join(OUTPUT, *relative.split('/'))
    assert digest(path) == expected, relative
    summary = json.load(open(path, encoding='utf-8'))
    binding = summary['binding']
    assert summary['status'] == 'pass' and summary['held_out_test_accessed'] is False
    assert summary['auxiliary_factor_losses'] == []
    assert set(summary['artifact_sha256']) == required_run_artifacts
    assert binding['project_commit'] == actual_commit
    assert binding['preserved_source_id'] == PRESERVED_SOURCE_ID
    assert binding['candidate_pool_sha256'] == POOL_SHA256
    run_root = os.path.dirname(path)
    for name, artifact_sha in summary['artifact_sha256'].items():
        assert digest(os.path.join(run_root, name)) == artifact_sha, (relative, name)
decision = json.load(open(os.path.join(OUTPUT, 'continuation_decision.json'), encoding='utf-8'))
assert decision['status'] == 'pass' and decision['held_out_test_accessed'] is False
assert manifest['continuation_selected'] == decision['selected_for_richer_stage']
run_metadata = {
    'status': 'pass', 'project_commit': actual_commit,
    'execution_protocol_sha256': EXPECTED_PROTOCOL_SHA256,
    'input_manifest_sha256': EXPECTED_INPUT_SHA256,
    'candidate_pool_sha256': POOL_SHA256,
    'pilot_manifest_sha256': digest(manifest_path),
    'continuation_selected': decision['selected_for_richer_stage'],
    'held_out_test_accessed': False, **versions,
}
with open(os.path.join(OUTPUT, 'run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2, sort_keys=True)
    handle.write('\n')
for path in (WORKTREE, SMOKE_OUTPUT):
    if os.path.exists(path):
        shutil.rmtree(path)
print({'pilot_manifest_sha256': run_metadata['pilot_manifest_sha256'], 'candidate_pool_sha256': POOL_SHA256, 'continuation_selected': decision['selected_for_richer_stage'], 'requirements': decision['requirements']})
print('TASK-TREATMENT PILOTS: PASS')

After the final PASS marker, save a Kaggle version with outputs. Preserve `/kaggle/working/task-aware-eeg2text-task-treatment-pilots` as a private dataset. A `false` continuation decision is a valid scientific result and must not be changed by retuning this frozen pilot.